# 03_dq: data quality checks on Silver

Runs every rule in `governance/dq_rules.yaml` against the Silver tables and writes:

| Table | One row per |
|---|---|
| `dq_results` | rule × table × column, per run: records checked, failures, PASS/FAIL, sample keys |
| `dq_exceptions` | failing record, per run: its key, the column and what is wrong |
| `dq_null_profile` | Silver column, per run: how many values are blank |

Nothing is fixed or removed here. Problems are reported to the data owner named on each
rule, who corrects them in the source system.

Attach **ll_silver** as the default lakehouse.

In [ ]:
run_id = ""  # blank: a timestamp, e.g. DQ20261122T101500
silver_db = "ll_silver"

## Contract and rules
Generated by `python -m fabric.build_notebooks` from the data dictionary and
`governance/dq_rules.yaml`.

In [ ]:
# --- generated: contract ---
SILVER_CONTRACT = {'silver_staff': {'key': ['staff_code'],
                  'source_system': 'MRI Qube PM, MRI Engage CRM, FixFlo',
                  'source_table': 'qube_staff, engage_negotiators, fixflo_agents',
                  'columns': [('staff_code', 'STRING'),
                              ('first_name', 'STRING'),
                              ('last_name', 'STRING'),
                              ('full_name', 'STRING'),
                              ('role', 'STRING'),
                              ('region', 'STRING'),
                              ('location', 'STRING'),
                              ('email', 'STRING'),
                              ('start_date', 'DATE'),
                              ('end_date', 'DATE'),
                              ('is_active', 'BOOLEAN'),
                              ('is_property_manager', 'BOOLEAN'),
                              ('is_negotiator', 'BOOLEAN'),
                              ('fixflo_agent_id', 'STRING'),
                              ('source_system', 'STRING'),
                              ('source_table', 'STRING'),
                              ('loaded_at', 'TIMESTAMP')],
                  'foreign_keys': []},
 'silver_block': {'key': ['block_ref'],
                  'source_system': 'MRI Qube PM, FixFlo',
                  'source_table': 'qube_blocks, fixflo_blocks',
                  'columns': [('block_ref', 'STRING'),
                              ('block_name', 'STRING'),
                              ('address_line1', 'STRING'),
                              ('town', 'STRING'),
                              ('county', 'STRING'),
                              ('postcode', 'STRING'),
                              ('region', 'STRING'),
                              ('latitude', 'DECIMAL(9,5)'),
                              ('longitude', 'DECIMAL(9,5)'),
                              ('floors', 'INT'),
                              ('built_year', 'INT'),
                              ('units', 'INT'),
                              ('fixflo_block_id', 'STRING'),
                              ('source_system', 'STRING'),
                              ('source_table', 'STRING'),
                              ('loaded_at', 'TIMESTAMP')],
                  'foreign_keys': []},
 'silver_property': {'key': ['property_ref'],
                     'source_system': 'MRI Qube PM, FixFlo',
                     'source_table': 'qube_properties, fixflo_properties',
                     'columns': [('property_ref', 'STRING'),
                                 ('block_ref', 'STRING'),
                                 ('address_line1', 'STRING'),
                                 ('address_line2', 'STRING'),
                                 ('town', 'STRING'),
                                 ('county', 'STRING'),
                                 ('postcode', 'STRING'),
                                 ('region', 'STRING'),
                                 ('latitude', 'DECIMAL(9,5)'),
                                 ('longitude', 'DECIMAL(9,5)'),
                                 ('property_type', 'STRING'),
                                 ('bedrooms', 'INT'),
                                 ('bathrooms', 'INT'),
                                 ('tenure', 'STRING'),
                                 ('department', 'STRING'),
                                 ('has_gas', 'BOOLEAN'),
                                 ('manager_staff_code', 'STRING'),
                                 ('acquisition_date', 'DATE'),
                                 ('acquisition_cost', 'DECIMAL(18,2)'),
                                 ('market_value', 'DECIMAL(18,2)'),
                                 ('market_rent_pcm', 'DECIMAL(18,2)'),
                                 ('property_status', 'STRING'),
                                 ('disposal_date', 'DATE'),
                                 ('disposal_value', 'DECIMAL(18,2)'),
                                 ('fixflo_property_id', 'STRING'),
                                 ('source_system', 'STRING'),
                                 ('source_table', 'STRING'),
                                 ('loaded_at', 'TIMESTAMP')],
                     'foreign_keys': [('block_ref', 'silver_block'),
                                      ('manager_staff_code', 'silver_staff')]},
 'silver_landlord': {'key': ['landlord_ref'],
                     'source_system': 'MRI Qube PM',
                     'source_table': 'qube_landlords, qube_landlord_bank_accounts',
                     'columns': [('landlord_ref', 'STRING'),
                                 ('landlord_name', 'STRING'),
                                 ('category', 'STRING'),
                                 ('email', 'STRING'),
                                 ('phone', 'STRING'),
                                 ('address_line1', 'STRING'),
                                 ('town', 'STRING'),
                                 ('county', 'STRING'),
                                 ('postcode', 'STRING'),
                                 ('created_date', 'DATE'),
                                 ('modified_date', 'DATE'),
                                 ('bank_account_name', 'STRING'),
                                 ('bank_sort_code', 'STRING'),
                                 ('bank_account_number', 'STRING'),
                                 ('source_system', 'STRING'),
                                 ('source_table', 'STRING'),
                                 ('loaded_at', 'TIMESTAMP')],
                     'foreign_keys': []},
 'silver_tenant': {'key': ['tenant_ref'],
                   'source_system': 'MRI Qube PM',
                   'source_table': 'qube_tenants',
                   'columns': [('tenant_ref', 'STRING'),
                               ('title', 'STRING'),
                               ('first_name', 'STRING'),
                               ('last_name', 'STRING'),
                               ('full_name', 'STRING'),
                               ('email', 'STRING'),
                               ('mobile', 'STRING'),
                               ('engage_applicant_id', 'STRING'),
                               ('right_to_rent_checked', 'BOOLEAN'),
                               ('right_to_rent_check_date', 'DATE'),
                               ('source_system', 'STRING'),
                               ('source_table', 'STRING'),
                               ('loaded_at', 'TIMESTAMP')],
                   'foreign_keys': [('engage_applicant_id', 'silver_applicant')]},
 'silver_applicant': {'key': ['applicant_id'],
                      'source_system': 'MRI Engage CRM',
                      'source_table': 'engage_applicants',
                      'columns': [('applicant_id', 'STRING'),
                                  ('applicant_type', 'STRING'),
                                  ('first_name', 'STRING'),
                                  ('last_name', 'STRING'),
                                  ('email', 'STRING'),
                                  ('mobile', 'STRING'),
                                  ('registered_date', 'DATE'),
                                  ('preferred_town', 'STRING'),
                                  ('min_bedrooms', 'INT'),
                                  ('max_budget', 'DECIMAL(18,2)'),
                                  ('status', 'STRING'),
                                  ('negotiator_code', 'STRING'),
                                  ('qube_tenant_ref', 'STRING'),
                                  ('source_system', 'STRING'),
                                  ('source_table', 'STRING'),
                                  ('loaded_at', 'TIMESTAMP')],
                      'foreign_keys': [('negotiator_code', 'silver_staff'),
                                       ('qube_tenant_ref', 'silver_tenant')]},
 'silver_tenancy': {'key': ['tenancy_ref'],
                    'source_system': 'MRI Qube PM',
                    'source_table': 'qube_tenancies',
                    'columns': [('tenancy_ref', 'STRING'),
                                ('property_ref', 'STRING'),
                                ('lead_tenant_ref', 'STRING'),
                                ('previous_tenancy_ref', 'STRING'),
                                ('tenancy_type', 'STRING'),
                                ('start_date', 'DATE'),
                                ('fixed_term_end_date', 'DATE'),
                                ('actual_end_date', 'DATE'),
                                ('tenancy_status', 'STRING'),
                                ('rent_pcm', 'DECIMAL(18,2)'),
                                ('rent_frequency', 'STRING'),
                                ('rent_due_day', 'DECIMAL(18,2)'),
                                ('ownership_share_pct', 'DECIMAL(18,2)'),
                                ('full_market_value', 'DECIMAL(18,2)'),
                                ('share_value', 'DECIMAL(18,2)'),
                                ('rent_rate_pct', 'DECIMAL(18,2)'),
                                ('rent_percentage_paid', 'DECIMAL(18,2)'),
                                ('deposit_amount', 'DECIMAL(18,2)'),
                                ('deposit_scheme', 'STRING'),
                                ('deposit_protected_date', 'DATE'),
                                ('deposit_status', 'STRING'),
                                ('closing_balance', 'DECIMAL(18,2)'),
                                ('last_updated_date', 'DATE'),
                                ('source_system', 'STRING'),
                                ('source_table', 'STRING'),
                                ('loaded_at', 'TIMESTAMP')],
                    'foreign_keys': [('property_ref', 'silver_property'),
                                     ('lead_tenant_ref', 'silver_tenant')]},
 'silver_tenancy_tenant': {'key': ['tenancy_ref', 'tenant_ref'],
                           'source_system': 'MRI Qube PM',
                           'source_table': 'qube_tenancy_tenants',
                           'columns': [('tenancy_ref', 'STRING'),
                                       ('tenant_ref', 'STRING'),
                                       ('is_lead_tenant', 'BOOLEAN'),
                                       ('rent_share_pct', 'DECIMAL(18,2)'),
                                       ('start_date', 'DATE'),
                                       ('end_date', 'DATE'),
                                       ('source_system', 'STRING'),
                                       ('source_table', 'STRING'),
                                       ('loaded_at', 'TIMESTAMP')],
                           'foreign_keys': [('tenant_ref', 'silver_tenant')]},
 'silver_ownership': {'key': ['owner_id'],
                      'source_system': 'MRI Qube PM',
                      'source_table': 'qube_property_owners',
                      'columns': [('owner_id', 'STRING'),
                                  ('property_ref', 'STRING'),
                                  ('owner_type', 'STRING'),
                                  ('landlord_ref', 'STRING'),
                                  ('tenant_ref', 'STRING'),
                                  ('percentage_share', 'DECIMAL(18,2)'),
                                  ('effective_from', 'DATE'),
                                  ('effective_to', 'DATE'),
                                  ('is_current', 'BOOLEAN'),
                                  ('source_system', 'STRING'),
                                  ('source_table', 'STRING'),
                                  ('loaded_at', 'TIMESTAMP')],
                      'foreign_keys': [('property_ref', 'silver_property'),
                                       ('landlord_ref', 'silver_landlord'),
                                       ('tenant_ref', 'silver_tenant')]},
 'silver_rent_transaction': {'key': ['transaction_id'],
                             'source_system': 'MRI Qube PM',
                             'source_table': 'qube_transactions',
                             'columns': [('transaction_id', 'STRING'),
                                         ('tenancy_ref', 'STRING'),
                                         ('property_ref', 'STRING'),
                                         ('tenant_ref', 'STRING'),
                                         ('transaction_date', 'DATE'),
                                         ('period_month', 'DATE'),
                                         ('transaction_type', 'STRING'),
                                         ('transaction_group', 'STRING'),
                                         ('amount', 'DECIMAL(18,2)'),
                                         ('payment_method', 'STRING'),
                                         ('reference', 'STRING'),
                                         ('ownership_share_pct', 'DECIMAL(18,2)'),
                                         ('source_system', 'STRING'),
                                         ('source_table', 'STRING'),
                                         ('loaded_at', 'TIMESTAMP')],
                             'foreign_keys': [('tenancy_ref', 'silver_tenancy'),
                                              ('property_ref', 'silver_property'),
                                              ('tenant_ref', 'silver_tenant')]},
 'silver_income_category': {'key': ['income_category'],
                            'source_system': 'MRI Qube PM',
                            'source_table': 'qube_income_categories',
                            'columns': [('income_category', 'STRING'),
                                        ('description', 'STRING'),
                                        ('priority_code', 'STRING'),
                                        ('billing_frequency', 'STRING'),
                                        ('source_system', 'STRING'),
                                        ('source_table', 'STRING'),
                                        ('loaded_at', 'TIMESTAMP')],
                            'foreign_keys': []},
 'silver_suspense_item': {'key': ['item_id'],
                          'source_system': 'MRI Qube PM',
                          'source_table': 'qube_suspense_items',
                          'columns': [('item_id', 'STRING'),
                                      ('item_type', 'STRING'),
                                      ('property_ref', 'STRING'),
                                      ('raised_date', 'DATE'),
                                      ('amount', 'DECIMAL(18,2)'),
                                      ('status', 'STRING'),
                                      ('cleared_date', 'DATE'),
                                      ('owner_staff_code', 'STRING'),
                                      ('notes', 'STRING'),
                                      ('source_system', 'STRING'),
                                      ('source_table', 'STRING'),
                                      ('loaded_at', 'TIMESTAMP')],
                          'foreign_keys': [('property_ref', 'silver_property'),
                                           ('owner_staff_code', 'silver_staff')]},
 'silver_supplier': {'key': ['supplier_code'],
                     'source_system': 'MRI Qube PM, FixFlo',
                     'source_table': 'qube_suppliers, fixflo_contractors',
                     'columns': [('supplier_code', 'STRING'),
                                 ('fixflo_contractor_id', 'STRING'),
                                 ('supplier_name', 'STRING'),
                                 ('trade', 'STRING'),
                                 ('region', 'STRING'),
                                 ('email', 'STRING'),
                                 ('phone', 'STRING'),
                                 ('gas_safe_number', 'STRING'),
                                 ('insurance_expiry', 'DATE'),
                                 ('is_active', 'BOOLEAN'),
                                 ('is_vat_registered', 'BOOLEAN'),
                                 ('source_system', 'STRING'),
                                 ('source_table', 'STRING'),
                                 ('loaded_at', 'TIMESTAMP')],
                     'foreign_keys': []},
 'silver_purchase_invoice': {'key': ['invoice_id'],
                             'source_system': 'MRI Qube PM',
                             'source_table': 'qube_purchase_invoices',
                             'columns': [('invoice_id', 'STRING'),
                                         ('supplier_code', 'STRING'),
                                         ('property_ref', 'STRING'),
                                         ('block_ref', 'STRING'),
                                         ('invoice_date', 'DATE'),
                                         ('category', 'STRING'),
                                         ('sub_category', 'STRING'),
                                         ('net_amount', 'DECIMAL(18,2)'),
                                         ('vat_amount', 'DECIMAL(18,2)'),
                                         ('gross_amount', 'DECIMAL(18,2)'),
                                         ('fixflo_issue_id', 'STRING'),
                                         ('status', 'STRING'),
                                         ('source_system', 'STRING'),
                                         ('source_table', 'STRING'),
                                         ('loaded_at', 'TIMESTAMP')],
                             'foreign_keys': [('supplier_code', 'silver_supplier'),
                                              ('property_ref', 'silver_property'),
                                              ('block_ref', 'silver_block')]},
 'silver_compliance_certificate': {'key': ['certificate_id'],
                                   'source_system': 'MRI Qube PM',
                                   'source_table': 'qube_compliance_certificates',
                                   'columns': [('certificate_id', 'STRING'),
                                               ('property_ref', 'STRING'),
                                               ('certificate_type', 'STRING'),
                                               ('issue_date', 'DATE'),
                                               ('expiry_date', 'DATE'),
                                               ('supplier_code', 'STRING'),
                                               ('outcome', 'STRING'),
                                               ('document_ref', 'STRING'),
                                               ('source_system', 'STRING'),
                                               ('source_table', 'STRING'),
                                               ('loaded_at', 'TIMESTAMP')],
                                   'foreign_keys': [('property_ref', 'silver_property'),
                                                    ('supplier_code', 'silver_supplier')]},
 'silver_repair_issue': {'key': ['issue_id'],
                         'source_system': 'FixFlo',
                         'source_table': 'fixflo_issues, fixflo_properties',
                         'columns': [('issue_id', 'STRING'),
                                     ('property_ref', 'STRING'),
                                     ('fixflo_property_id', 'STRING'),
                                     ('block_ref', 'STRING'),
                                     ('tenancy_ref', 'STRING'),
                                     ('issue_type', 'STRING'),
                                     ('title', 'STRING'),
                                     ('fault_title', 'STRING'),
                                     ('fault_category', 'STRING'),
                                     ('priority', 'INT'),
                                     ('is_emergency', 'BOOLEAN'),
                                     ('is_communal', 'BOOLEAN'),
                                     ('status', 'STRING'),
                                     ('created_at', 'TIMESTAMP'),
                                     ('status_changed_at', 'TIMESTAMP'),
                                     ('close_reason', 'STRING'),
                                     ('close_reason_description', 'STRING'),
                                     ('job_id', 'STRING'),
                                     ('source_system', 'STRING'),
                                     ('source_table', 'STRING'),
                                     ('loaded_at', 'TIMESTAMP')],
                         'foreign_keys': [('property_ref', 'silver_property'),
                                          ('block_ref', 'silver_block'),
                                          ('tenancy_ref', 'silver_tenancy'),
                                          ('job_id', 'silver_repair_job')]},
 'silver_repair_job': {'key': ['job_id'],
                       'source_system': 'FixFlo',
                       'source_table': 'fixflo_jobs, fixflo_job_completion_details',
                       'columns': [('job_id', 'STRING'),
                                   ('issue_id', 'STRING'),
                                   ('supplier_code', 'STRING'),
                                   ('fixflo_contractor_id', 'STRING'),
                                   ('works_order_number', 'STRING'),
                                   ('created_at', 'TIMESTAMP'),
                                   ('start_at', 'TIMESTAMP'),
                                   ('completed_at', 'TIMESTAMP'),
                                   ('job_duration', 'STRING'),
                                   ('appointment', 'STRING'),
                                   ('payable_net', 'DECIMAL(18,2)'),
                                   ('payable_tax', 'DECIMAL(18,2)'),
                                   ('payable_gross', 'DECIMAL(18,2)'),
                                   ('quoted_gross', 'DECIMAL(18,2)'),
                                   ('invoice_number', 'STRING'),
                                   ('completion_date', 'DATE'),
                                   ('feedback', 'STRING'),
                                   ('source_system', 'STRING'),
                                   ('source_table', 'STRING'),
                                   ('loaded_at', 'TIMESTAMP')],
                       'foreign_keys': [('issue_id', 'silver_repair_issue'),
                                        ('supplier_code', 'silver_supplier')]},
 'silver_repair_quote': {'key': ['quote_id'],
                         'source_system': 'FixFlo',
                         'source_table': 'fixflo_quotes',
                         'columns': [('quote_id', 'STRING'),
                                     ('issue_id', 'STRING'),
                                     ('supplier_code', 'STRING'),
                                     ('status', 'STRING'),
                                     ('payable_net', 'DECIMAL(18,2)'),
                                     ('payable_tax', 'DECIMAL(18,2)'),
                                     ('payable_gross', 'DECIMAL(18,2)'),
                                     ('source_system', 'STRING'),
                                     ('source_table', 'STRING'),
                                     ('loaded_at', 'TIMESTAMP')],
                         'foreign_keys': [('issue_id', 'silver_repair_issue'),
                                          ('supplier_code', 'silver_supplier')]},
 'silver_listing': {'key': ['listing_id'],
                    'source_system': 'MRI Engage CRM',
                    'source_table': 'engage_listings',
                    'columns': [('listing_id', 'STRING'),
                                ('property_ref', 'STRING'),
                                ('listing_type', 'STRING'),
                                ('listed_date', 'DATE'),
                                ('asking_rent_pcm', 'DECIMAL(18,2)'),
                                ('asking_price', 'DECIMAL(18,2)'),
                                ('listing_status', 'STRING'),
                                ('agreed_date', 'DATE'),
                                ('completion_date', 'DATE'),
                                ('achieved_rent_pcm', 'DECIMAL(18,2)'),
                                ('achieved_price', 'DECIMAL(18,2)'),
                                ('negotiator_code', 'STRING'),
                                ('withdrawn_date', 'DATE'),
                                ('source_system', 'STRING'),
                                ('source_table', 'STRING'),
                                ('loaded_at', 'TIMESTAMP')],
                    'foreign_keys': [('property_ref', 'silver_property'),
                                     ('negotiator_code', 'silver_staff')]},
 'silver_viewing': {'key': ['viewing_id'],
                    'source_system': 'MRI Engage CRM',
                    'source_table': 'engage_viewings',
                    'columns': [('viewing_id', 'STRING'),
                                ('listing_id', 'STRING'),
                                ('applicant_id', 'STRING'),
                                ('viewing_date', 'DATE'),
                                ('outcome', 'STRING'),
                                ('negotiator_code', 'STRING'),
                                ('source_system', 'STRING'),
                                ('source_table', 'STRING'),
                                ('loaded_at', 'TIMESTAMP')],
                    'foreign_keys': [('listing_id', 'silver_listing'),
                                     ('applicant_id', 'silver_applicant'),
                                     ('negotiator_code', 'silver_staff')]},
 'silver_offer': {'key': ['offer_id'],
                  'source_system': 'MRI Engage CRM',
                  'source_table': 'engage_offers',
                  'columns': [('offer_id', 'STRING'),
                              ('listing_id', 'STRING'),
                              ('applicant_id', 'STRING'),
                              ('offer_date', 'DATE'),
                              ('offer_amount', 'DECIMAL(18,2)'),
                              ('offer_status', 'STRING'),
                              ('source_system', 'STRING'),
                              ('source_table', 'STRING'),
                              ('loaded_at', 'TIMESTAMP')],
                  'foreign_keys': [('listing_id', 'silver_listing'),
                                   ('applicant_id', 'silver_applicant')]},
 'silver_calendar': {'key': ['date'],
                     'source_system': 'Reference',
                     'source_table': 'dim_calendar',
                     'columns': [('date', 'DATE'),
                                 ('year', 'INT'),
                                 ('quarter', 'STRING'),
                                 ('month_number', 'STRING'),
                                 ('month_name', 'STRING'),
                                 ('month_short', 'STRING'),
                                 ('year_month', 'STRING'),
                                 ('year_quarter', 'STRING'),
                                 ('weekday_name', 'STRING'),
                                 ('is_weekend', 'BOOLEAN'),
                                 ('financial_year', 'STRING'),
                                 ('source_system', 'STRING'),
                                 ('source_table', 'STRING'),
                                 ('loaded_at', 'TIMESTAMP')],
                     'foreign_keys': []}}
# --- end generated ---

In [ ]:
# --- generated: dq_rules ---
DQ_RULES = {'domains': {'Housing management': {'owner': 'Head of Housing Management',
                                    'steward': 'Housing data steward',
                                    'tables': ['silver_property',
                                               'silver_block',
                                               'silver_tenancy',
                                               'silver_tenancy_tenant',
                                               'silver_tenant',
                                               'silver_ownership',
                                               'silver_landlord']},
             'Finance': {'owner': 'Head of Finance',
                         'steward': 'Finance systems analyst',
                         'tables': ['silver_rent_transaction',
                                    'silver_income_category',
                                    'silver_suspense_item',
                                    'silver_purchase_invoice']},
             'Repairs': {'owner': 'Head of Repairs and Maintenance',
                         'steward': 'Repairs coordinator',
                         'tables': ['silver_repair_issue',
                                    'silver_repair_job',
                                    'silver_repair_quote',
                                    'silver_supplier']},
             'Compliance': {'owner': 'Compliance Manager',
                            'steward': 'Compliance administrator',
                            'tables': ['silver_compliance_certificate']},
             'Lettings and sales': {'owner': 'Head of Lettings and Sales',
                                    'steward': 'Lettings administrator',
                                    'tables': ['silver_applicant',
                                               'silver_listing',
                                               'silver_viewing',
                                               'silver_offer']},
             'People': {'owner': 'HR Manager',
                        'steward': 'HR systems administrator',
                        'tables': ['silver_staff']},
             'Reference data': {'owner': 'Head of Data',
                                'steward': 'Data steward',
                                'tables': ['silver_calendar']}},
 'rules': [{'id': 'DQ001',
            'name': 'Primary key is unique and not blank',
            'dimension': 'uniqueness',
            'severity': 'high',
            'owner': 'table owner',
            'table': 'all',
            'column': 'key',
            'check': 'primary_key'},
           {'id': 'DQ010',
            'name': 'No duplicate properties (same address and postcode on two records)',
            'dimension': 'uniqueness',
            'severity': 'high',
            'owner': 'Head of Housing Management',
            'table': 'silver_property',
            'column': 'address_line1, address_line2, postcode',
            'check': 'sql',
            'sql': 'SELECT property_ref AS record_key,\n'
                   "       'address_line1, postcode' AS column_name,\n"
                   "       concat('same address as ', n - 1, ' other record(s): ',\n"
                   "              concat_ws(', ', address_line1, address_line2, postcode),\n"
                   "              ' (status ', property_status, ')') AS detail\n"
                   'FROM (\n'
                   '    SELECT p.*,\n'
                   '           COUNT(*) OVER (PARTITION BY upper(trim(address_line1)),\n'
                   '                                       upper(trim(coalesce(address_line2, '
                   "''))),\n"
                   "                                       upper(replace(postcode, ' ', ''))) "
                   'AS n\n'
                   '    FROM {db}.silver_property p\n'
                   '    WHERE address_line1 IS NOT NULL AND postcode IS NOT NULL\n'
                   ') x\n'
                   'WHERE n > 1\n'},
           {'id': 'DQ011',
            'name': 'A mobile number belongs to one tenant only',
            'dimension': 'uniqueness',
            'severity': 'medium',
            'owner': 'Head of Housing Management',
            'table': 'silver_tenant',
            'column': 'mobile',
            'check': 'sql',
            'sql': "SELECT tenant_ref AS record_key, 'mobile' AS column_name,\n"
                   "       concat('mobile ', mobile, ' is on ', n, ' tenant records') AS "
                   'detail\n'
                   'FROM (\n'
                   '    SELECT tenant_ref, mobile,\n'
                   "           COUNT(*) OVER (PARTITION BY replace(mobile, ' ', '')) AS n\n"
                   '    FROM {db}.silver_tenant\n'
                   '    WHERE mobile IS NOT NULL\n'
                   ') x\n'
                   'WHERE n > 1\n'},
           {'id': 'DQ012',
            'name': 'At most one accepted offer per listing',
            'dimension': 'uniqueness',
            'severity': 'high',
            'owner': 'Head of Lettings and Sales',
            'table': 'silver_offer',
            'column': 'offer_status',
            'check': 'sql',
            'sql': "SELECT offer_id AS record_key, 'offer_status' AS column_name,\n"
                   "       concat('listing ', listing_id, ' has ', n, ' accepted offers') AS "
                   'detail\n'
                   'FROM (\n'
                   '    SELECT offer_id, listing_id, COUNT(*) OVER (PARTITION BY listing_id) '
                   'AS n\n'
                   '    FROM {db}.silver_offer\n'
                   "    WHERE offer_status = 'Accepted'\n"
                   ') x\n'
                   'WHERE n > 1\n'},
           {'id': 'DQ002',
            'name': 'Every link in the data dictionary points at an existing record',
            'dimension': 'referential integrity',
            'severity': 'high',
            'owner': 'table owner',
            'table': 'all',
            'column': 'foreign keys',
            'check': 'foreign_key'},
           {'id': 'DQ020',
            'name': 'Mandatory property fields are filled',
            'dimension': 'completeness',
            'severity': 'high',
            'owner': 'Head of Housing Management',
            'table': 'silver_property',
            'column': 'address_line1, town, postcode, region, property_type, tenure, '
                      'department, property_status, manager_staff_code, acquisition_date',
            'check': 'not_null',
            'columns': ['address_line1',
                        'town',
                        'postcode',
                        'region',
                        'property_type',
                        'tenure',
                        'department',
                        'property_status',
                        'manager_staff_code',
                        'acquisition_date']},
           {'id': 'DQ021',
            'name': 'Mandatory tenancy fields are filled',
            'dimension': 'completeness',
            'severity': 'high',
            'owner': 'Head of Housing Management',
            'table': 'silver_tenancy',
            'column': 'property_ref, lead_tenant_ref, tenancy_type, tenancy_status, '
                      'start_date, rent_pcm',
            'check': 'not_null',
            'columns': ['property_ref',
                        'lead_tenant_ref',
                        'tenancy_type',
                        'tenancy_status',
                        'start_date',
                        'rent_pcm']},
           {'id': 'DQ022',
            'name': 'Mandatory rent ledger fields are filled (incl. a known transaction type)',
            'dimension': 'completeness',
            'severity': 'high',
            'owner': 'Head of Finance',
            'table': 'silver_rent_transaction',
            'column': 'tenancy_ref, property_ref, transaction_date, period_month, '
                      'transaction_type, transaction_group, amount',
            'check': 'not_null',
            'columns': ['tenancy_ref',
                        'property_ref',
                        'transaction_date',
                        'period_month',
                        'transaction_type',
                        'transaction_group',
                        'amount']},
           {'id': 'DQ023',
            'name': 'Tenants have a first and last name',
            'dimension': 'completeness',
            'severity': 'medium',
            'owner': 'Head of Housing Management',
            'table': 'silver_tenant',
            'column': 'first_name, last_name',
            'check': 'not_null',
            'columns': ['first_name', 'last_name']},
           {'id': 'DQ024',
            'name': 'Certificates have type, dates and outcome',
            'dimension': 'completeness',
            'severity': 'high',
            'owner': 'Compliance Manager',
            'table': 'silver_compliance_certificate',
            'column': 'property_ref, certificate_type, issue_date, expiry_date, outcome',
            'check': 'not_null',
            'columns': ['property_ref',
                        'certificate_type',
                        'issue_date',
                        'expiry_date',
                        'outcome']},
           {'id': 'DQ025',
            'name': 'Repair issues have a property, status and reported date',
            'dimension': 'completeness',
            'severity': 'medium',
            'owner': 'Head of Repairs and Maintenance',
            'table': 'silver_repair_issue',
            'column': 'property_ref, status, created_at',
            'check': 'not_null',
            'columns': ['property_ref', 'status', 'created_at']},
           {'id': 'DQ003',
            'name': 'Values convert to their dictionary type',
            'dimension': 'validity',
            'severity': 'high',
            'owner': 'table owner',
            'table': 'all',
            'column': 'typed columns',
            'check': 'type_converts'},
           {'id': 'DQ030',
            'name': 'Tenant email is a valid email address',
            'dimension': 'validity',
            'severity': 'medium',
            'owner': 'Head of Housing Management',
            'table': 'silver_tenant',
            'column': 'email',
            'check': 'pattern',
            'regex': '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$'},
           {'id': 'DQ031',
            'name': 'Applicant email is a valid email address',
            'dimension': 'validity',
            'severity': 'low',
            'owner': 'Head of Lettings and Sales',
            'table': 'silver_applicant',
            'column': 'email',
            'check': 'pattern',
            'regex': '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$'},
           {'id': 'DQ032',
            'name': 'Staff email is a valid email address',
            'dimension': 'validity',
            'severity': 'medium',
            'owner': 'HR Manager',
            'table': 'silver_staff',
            'column': 'email',
            'check': 'pattern',
            'regex': '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$'},
           {'id': 'DQ033',
            'name': 'Property postcode is a valid UK postcode',
            'dimension': 'validity',
            'severity': 'medium',
            'owner': 'Head of Housing Management',
            'table': 'silver_property',
            'column': 'postcode',
            'check': 'pattern',
            'regex': '^[A-Z]{1,2}[0-9][A-Z0-9]? ?[0-9][A-Z]{2}$'},
           {'id': 'DQ034',
            'name': 'Tenant mobile is a valid UK mobile number',
            'dimension': 'validity',
            'severity': 'low',
            'owner': 'Head of Housing Management',
            'table': 'silver_tenant',
            'column': 'mobile',
            'check': 'pattern',
            'regex': '^(07[0-9]{3} ?[0-9]{6}|\\+44 ?7[0-9]{3} ?[0-9]{6})$'},
           {'id': 'DQ035',
            'name': 'Applicant mobile is a valid UK mobile number',
            'dimension': 'validity',
            'severity': 'low',
            'owner': 'Head of Lettings and Sales',
            'table': 'silver_applicant',
            'column': 'mobile',
            'check': 'pattern',
            'regex': '^(07[0-9]{3} ?[0-9]{6}|\\+44 ?7[0-9]{3} ?[0-9]{6})$'},
           {'id': 'DQ036',
            'name': 'Tenancy status is a known value',
            'dimension': 'validity',
            'severity': 'high',
            'owner': 'Head of Housing Management',
            'table': 'silver_tenancy',
            'column': 'tenancy_status',
            'check': 'allowed_values',
            'values': ['Active', 'Periodic', 'Renewed', 'Ended']},
           {'id': 'DQ037',
            'name': 'Property status is a known value',
            'dimension': 'validity',
            'severity': 'high',
            'owner': 'Head of Housing Management',
            'table': 'silver_property',
            'column': 'property_status',
            'check': 'allowed_values',
            'values': ['Let', 'Void', 'Sold']},
           {'id': 'DQ038',
            'name': 'Tenure is a known value',
            'dimension': 'validity',
            'severity': 'high',
            'owner': 'Head of Housing Management',
            'table': 'silver_property',
            'column': 'tenure',
            'check': 'allowed_values',
            'values': ['PRS', 'Shared Ownership', 'Open Market Sale']},
           {'id': 'DQ039',
            'name': 'Repair issue status is a FixFlo status',
            'dimension': 'validity',
            'severity': 'medium',
            'owner': 'Head of Repairs and Maintenance',
            'table': 'silver_repair_issue',
            'column': 'status',
            'check': 'allowed_values',
            'values': ['Reported',
                       'JobAwarded',
                       'AwaitingJobCompletion',
                       'JobCompleted',
                       'Closed']},
           {'id': 'DQ040',
            'name': 'Certificate type is a known value',
            'dimension': 'validity',
            'severity': 'high',
            'owner': 'Compliance Manager',
            'table': 'silver_compliance_certificate',
            'column': 'certificate_type',
            'check': 'allowed_values',
            'values': ['Gas Safety Certificate', 'EICR', 'EPC']},
           {'id': 'DQ041',
            'name': 'Transaction type is a known value',
            'dimension': 'validity',
            'severity': 'high',
            'owner': 'Head of Finance',
            'table': 'silver_rent_transaction',
            'column': 'transaction_type',
            'check': 'allowed_values',
            'values': ['Rent charge',
                       'Service charge',
                       'Ground rent',
                       'Tenant recharge invoice',
                       'Receipt',
                       'Deposit received',
                       'Deposit returned',
                       'Deposit deduction']},
           {'id': 'DQ042',
            'name': 'Ownership share is between 0 and 100%',
            'dimension': 'validity',
            'severity': 'high',
            'owner': 'Head of Housing Management',
            'table': 'silver_ownership',
            'column': 'percentage_share',
            'check': 'range',
            'min': 0,
            'max': 100},
           {'id': 'DQ043',
            'name': 'Shared Ownership share on the tenancy is between 0 and 100%',
            'dimension': 'validity',
            'severity': 'high',
            'owner': 'Head of Housing Management',
            'table': 'silver_tenancy',
            'column': 'ownership_share_pct',
            'check': 'range',
            'min': 0,
            'max': 100},
           {'id': 'DQ044',
            'name': 'Monthly rent is above zero and not implausibly high (over £10,000)',
            'dimension': 'validity',
            'severity': 'high',
            'owner': 'Head of Finance',
            'table': 'silver_tenancy',
            'column': 'rent_pcm',
            'check': 'range',
            'min': 0.01,
            'max': 10000},
           {'id': 'DQ045',
            'name': 'Transaction dates are between Jan 2021 and today',
            'dimension': 'validity',
            'severity': 'high',
            'owner': 'Head of Finance',
            'table': 'silver_rent_transaction',
            'column': 'transaction_date',
            'check': 'date_window',
            'columns': ['transaction_date'],
            'min': '2021-01-01'},
           {'id': 'DQ046',
            'name': 'Tenancy start dates are between Jan 2021 and today',
            'dimension': 'validity',
            'severity': 'medium',
            'owner': 'Head of Housing Management',
            'table': 'silver_tenancy',
            'column': 'start_date',
            'check': 'date_window',
            'columns': ['start_date'],
            'min': '2021-01-01'},
           {'id': 'DQ047',
            'name': 'Acquisition dates are between Jan 2021 and today',
            'dimension': 'validity',
            'severity': 'medium',
            'owner': 'Head of Housing Management',
            'table': 'silver_property',
            'column': 'acquisition_date',
            'check': 'date_window',
            'columns': ['acquisition_date'],
            'min': '2021-01-01'},
           {'id': 'DQ048',
            'name': 'Certificate issue dates are between Jan 2021 and today',
            'dimension': 'validity',
            'severity': 'high',
            'owner': 'Compliance Manager',
            'table': 'silver_compliance_certificate',
            'column': 'issue_date',
            'check': 'date_window',
            'columns': ['issue_date'],
            'min': '2021-01-01'},
           {'id': 'DQ049',
            'name': 'Repairs were reported between Jan 2021 and today',
            'dimension': 'validity',
            'severity': 'medium',
            'owner': 'Head of Repairs and Maintenance',
            'table': 'silver_repair_issue',
            'column': 'created_at',
            'check': 'date_window',
            'columns': ['created_at'],
            'min': '2021-01-01'},
           {'id': 'DQ060',
            'name': 'Tenancy ends on or after it starts',
            'dimension': 'consistency',
            'severity': 'high',
            'owner': 'Head of Housing Management',
            'table': 'silver_tenancy',
            'column': 'start_date, actual_end_date, fixed_term_end_date',
            'check': 'sql',
            'sql': "SELECT tenancy_ref AS record_key, 'start_date / end dates' AS "
                   'column_name,\n'
                   "       concat('starts ', start_date, ' but ends ',\n"
                   '              coalesce(actual_end_date, fixed_term_end_date)) AS detail\n'
                   'FROM {db}.silver_tenancy\n'
                   'WHERE actual_end_date < start_date OR fixed_term_end_date < start_date\n'},
           {'id': 'DQ061',
            'name': 'Ended tenancies have an end date; live ones do not',
            'dimension': 'consistency',
            'severity': 'medium',
            'owner': 'Head of Housing Management',
            'table': 'silver_tenancy',
            'column': 'tenancy_status, actual_end_date',
            'check': 'sql',
            'sql': "SELECT tenancy_ref AS record_key, 'actual_end_date' AS column_name,\n"
                   "       concat(tenancy_status, CASE WHEN actual_end_date IS NULL THEN ' "
                   "with no end date'\n"
                   "                                   ELSE concat(' but ended ', "
                   'actual_end_date) END) AS detail\n'
                   'FROM {db}.silver_tenancy\n'
                   "WHERE (tenancy_status IN ('Ended', 'Renewed') AND actual_end_date IS "
                   'NULL)\n'
                   "   OR (tenancy_status IN ('Active', 'Periodic') AND actual_end_date IS "
                   'NOT NULL)\n'},
           {'id': 'DQ062',
            'name': 'No overlapping tenancies on a property',
            'dimension': 'consistency',
            'severity': 'high',
            'owner': 'Head of Housing Management',
            'table': 'silver_tenancy',
            'column': 'start_date, actual_end_date',
            'check': 'sql',
            'sql': "SELECT a.tenancy_ref AS record_key, 'start_date, actual_end_date' AS "
                   'column_name,\n'
                   "       concat('overlaps tenancy ', b.tenancy_ref, ' on property ', "
                   'a.property_ref) AS detail\n'
                   'FROM {db}.silver_tenancy a\n'
                   'JOIN {db}.silver_tenancy b\n'
                   '  ON a.property_ref = b.property_ref AND a.tenancy_ref <> b.tenancy_ref\n'
                   "WHERE a.start_date <= coalesce(b.actual_end_date, DATE '9999-12-31')\n"
                   "  AND b.start_date <= coalesce(a.actual_end_date, DATE '9999-12-31')\n"},
           {'id': 'DQ063',
            'name': 'Renewals follow on from the previous tenancy (same home, next day)',
            'dimension': 'consistency',
            'severity': 'medium',
            'owner': 'Head of Housing Management',
            'table': 'silver_tenancy',
            'column': 'previous_tenancy_ref',
            'check': 'sql',
            'sql': "SELECT t.tenancy_ref AS record_key, 'previous_tenancy_ref' AS "
                   'column_name,\n'
                   '       CASE WHEN p.tenancy_ref IS NULL\n'
                   "                THEN concat('previous tenancy ', t.previous_tenancy_ref, "
                   "' not found')\n"
                   '            WHEN p.property_ref <> t.property_ref\n'
                   "                THEN concat('previous tenancy is on ', p.property_ref)\n"
                   "            ELSE concat('starts ', t.start_date, ' but previous ended ',\n"
                   '                        coalesce(CAST(p.actual_end_date AS STRING), '
                   "'never'))\n"
                   '       END AS detail\n'
                   'FROM {db}.silver_tenancy t\n'
                   'LEFT JOIN {db}.silver_tenancy p ON p.tenancy_ref = '
                   't.previous_tenancy_ref\n'
                   'WHERE t.previous_tenancy_ref IS NOT NULL\n'
                   '  AND (p.tenancy_ref IS NULL\n'
                   '       OR p.property_ref <> t.property_ref\n'
                   '       OR p.actual_end_date IS NULL\n'
                   '       OR t.start_date <> date_add(p.actual_end_date, 1))\n',
            'checked_sql': 'SELECT COUNT(*) FROM {db}.silver_tenancy WHERE '
                           'previous_tenancy_ref IS NOT NULL'},
           {'id': 'DQ064',
            'name': 'Property status matches its tenancies (Let = one live tenancy; Void or '
                    'Sold = none)',
            'dimension': 'consistency',
            'severity': 'high',
            'owner': 'Head of Housing Management',
            'table': 'silver_property',
            'column': 'property_status',
            'check': 'sql',
            'sql': 'WITH live AS (\n'
                   '    SELECT property_ref, COUNT(*) AS n\n'
                   '    FROM {db}.silver_tenancy\n'
                   "    WHERE tenancy_status IN ('Active', 'Periodic')\n"
                   '    GROUP BY property_ref\n'
                   ')\n'
                   "SELECT p.property_ref AS record_key, 'property_status' AS column_name,\n"
                   "       concat(p.property_status, ' with ', coalesce(l.n, 0), ' live "
                   "tenancies') AS detail\n"
                   'FROM {db}.silver_property p\n'
                   'LEFT JOIN live l ON l.property_ref = p.property_ref\n'
                   "WHERE (p.property_status = 'Let' AND coalesce(l.n, 0) <> 1)\n"
                   "   OR (p.property_status IN ('Void', 'Sold') AND coalesce(l.n, 0) > 0)\n"},
           {'id': 'DQ065',
            'name': 'No charges, repairs, certificates or invoices after a home was sold',
            'dimension': 'consistency',
            'severity': 'high',
            'owner': 'Head of Housing Management',
            'table': 'silver_property',
            'column': 'disposal_date',
            'check': 'sql',
            'sql': 'WITH sold AS (\n'
                   '    SELECT property_ref, disposal_date FROM {db}.silver_property\n'
                   '    WHERE disposal_date IS NOT NULL\n'
                   '),\n'
                   'after_sale AS (\n'
                   "    SELECT t.property_ref, 'rent charges' AS what, COUNT(*) AS n\n"
                   '    FROM {db}.silver_rent_transaction t JOIN sold s ON s.property_ref = '
                   't.property_ref\n'
                   "    WHERE t.transaction_group = 'Charge' AND t.transaction_date > "
                   's.disposal_date\n'
                   '    GROUP BY t.property_ref\n'
                   '    UNION ALL\n'
                   "    SELECT i.property_ref, 'repairs', COUNT(*)\n"
                   '    FROM {db}.silver_repair_issue i JOIN sold s ON s.property_ref = '
                   'i.property_ref\n'
                   '    WHERE CAST(i.created_at AS DATE) > s.disposal_date\n'
                   '    GROUP BY i.property_ref\n'
                   '    UNION ALL\n'
                   "    SELECT c.property_ref, 'certificates', COUNT(*)\n"
                   '    FROM {db}.silver_compliance_certificate c JOIN sold s ON '
                   's.property_ref = c.property_ref\n'
                   '    WHERE c.issue_date > s.disposal_date\n'
                   '    GROUP BY c.property_ref\n'
                   '    UNION ALL\n'
                   "    SELECT v.property_ref, 'supplier invoices', COUNT(*)\n"
                   '    FROM {db}.silver_purchase_invoice v JOIN sold s ON s.property_ref = '
                   'v.property_ref\n'
                   '    WHERE v.invoice_date > s.disposal_date\n'
                   '    GROUP BY v.property_ref\n'
                   ')\n'
                   "SELECT property_ref AS record_key, 'disposal_date' AS column_name,\n"
                   "       concat(n, ' ', what, ' dated after the sale') AS detail\n"
                   'FROM after_sale\n',
            'checked_sql': 'SELECT COUNT(*) FROM {db}.silver_property WHERE disposal_date IS '
                           'NOT NULL'},
           {'id': 'DQ066',
            'name': 'Current ownership shares of a home add up to 100%',
            'dimension': 'consistency',
            'severity': 'high',
            'owner': 'Head of Housing Management',
            'table': 'silver_ownership',
            'column': 'percentage_share',
            'check': 'sql',
            'sql': "SELECT property_ref AS record_key, 'percentage_share' AS column_name,\n"
                   "       concat('current shares add up to ', total, '%') AS detail\n"
                   'FROM (\n'
                   '    SELECT property_ref, SUM(percentage_share) AS total\n'
                   '    FROM {db}.silver_ownership\n'
                   '    WHERE is_current\n'
                   '    GROUP BY property_ref\n'
                   ') x\n'
                   'WHERE total <> 100\n',
            'checked_sql': 'SELECT COUNT(DISTINCT property_ref) FROM {db}.silver_ownership '
                           'WHERE is_current'},
           {'id': 'DQ067',
            'name': 'Shared Ownership share value and rent percentage agree with the share',
            'dimension': 'consistency',
            'severity': 'medium',
            'owner': 'Head of Finance',
            'table': 'silver_tenancy',
            'column': 'share_value, rent_percentage_paid',
            'check': 'sql',
            'sql': "SELECT tenancy_ref AS record_key, 'share_value, rent_percentage_paid' AS "
                   'column_name,\n'
                   "       concat('share ', ownership_share_pct, '% of ', full_market_value,\n"
                   "              ' but share value ', share_value,\n"
                   "              ' and rent paid on ', rent_percentage_paid, '%') AS detail\n"
                   'FROM {db}.silver_tenancy\n'
                   'WHERE ownership_share_pct IS NOT NULL\n'
                   '  AND (abs(share_value - ownership_share_pct / 100 * full_market_value) > '
                   '1\n'
                   '       OR abs(rent_percentage_paid - (100 - ownership_share_pct)) > '
                   '0.01)\n',
            'checked_sql': 'SELECT COUNT(*) FROM {db}.silver_tenancy WHERE '
                           'ownership_share_pct IS NOT NULL'},
           {'id': 'DQ068',
            'name': "Closing balance equals the tenancy's ledger (deposits held excluded)",
            'dimension': 'consistency',
            'severity': 'high',
            'owner': 'Head of Finance',
            'table': 'silver_tenancy',
            'column': 'closing_balance',
            'check': 'sql',
            'sql': 'WITH ledger AS (\n'
                   '    SELECT tenancy_ref, SUM(amount) AS total\n'
                   '    FROM {db}.silver_rent_transaction\n'
                   "    WHERE transaction_type NOT IN ('Deposit received', 'Deposit "
                   "returned')\n"
                   '    GROUP BY tenancy_ref\n'
                   ')\n'
                   "SELECT t.tenancy_ref AS record_key, 'closing_balance' AS column_name,\n"
                   "       concat('closing balance ', coalesce(CAST(t.closing_balance AS "
                   "STRING), 'blank'),\n"
                   "              ' but the ledger adds up to ', coalesce(l.total, 0)) AS "
                   'detail\n'
                   'FROM {db}.silver_tenancy t\n'
                   'LEFT JOIN ledger l ON l.tenancy_ref = t.tenancy_ref\n'
                   'WHERE abs(coalesce(t.closing_balance, 0) - coalesce(l.total, 0)) > '
                   '0.005\n'},
           {'id': 'DQ069',
            'name': "No implausible balances (in credit, or more than 4 months' rent owed)",
            'dimension': 'consistency',
            'severity': 'medium',
            'owner': 'Head of Finance',
            'table': 'silver_tenancy',
            'column': 'closing_balance',
            'check': 'sql',
            'sql': "SELECT tenancy_ref AS record_key, 'closing_balance' AS column_name,\n"
                   "       concat('balance ', closing_balance, ' on rent of ', rent_pcm, ' a "
                   "month') AS detail\n"
                   'FROM {db}.silver_tenancy\n'
                   'WHERE closing_balance < 0 OR closing_balance > 4 * rent_pcm\n'},
           {'id': 'DQ070',
            'name': 'Charges are positive and payments negative',
            'dimension': 'consistency',
            'severity': 'high',
            'owner': 'Head of Finance',
            'table': 'silver_rent_transaction',
            'column': 'amount',
            'check': 'sql',
            'sql': "SELECT transaction_id AS record_key, 'amount' AS column_name,\n"
                   "       concat(transaction_type, ' of ', amount) AS detail\n"
                   'FROM {db}.silver_rent_transaction\n'
                   "WHERE (transaction_group = 'Charge' AND amount <= 0)\n"
                   "   OR (transaction_group = 'Payment' AND amount >= 0)\n"},
           {'id': 'DQ071',
            'name': 'Repair job net + tax = gross',
            'dimension': 'consistency',
            'severity': 'medium',
            'owner': 'Head of Repairs and Maintenance',
            'table': 'silver_repair_job',
            'column': 'payable_net, payable_tax, payable_gross',
            'check': 'sql',
            'sql': "SELECT job_id AS record_key, 'payable_gross' AS column_name,\n"
                   "       concat(payable_net, ' + ', payable_tax, ' <> ', payable_gross) AS "
                   'detail\n'
                   'FROM {db}.silver_repair_job\n'
                   'WHERE abs(payable_net + payable_tax - payable_gross) > 0.01\n'},
           {'id': 'DQ072',
            'name': 'Supplier invoice net + VAT = gross',
            'dimension': 'consistency',
            'severity': 'medium',
            'owner': 'Head of Finance',
            'table': 'silver_purchase_invoice',
            'column': 'net_amount, vat_amount, gross_amount',
            'check': 'sql',
            'sql': "SELECT invoice_id AS record_key, 'gross_amount' AS column_name,\n"
                   "       concat(net_amount, ' + ', vat_amount, ' <> ', gross_amount) AS "
                   'detail\n'
                   'FROM {db}.silver_purchase_invoice\n'
                   'WHERE abs(net_amount + vat_amount - gross_amount) > 0.01\n'},
           {'id': 'DQ073',
            'name': 'Repairs are only on rental (PRS) homes',
            'dimension': 'consistency',
            'severity': 'medium',
            'owner': 'Head of Repairs and Maintenance',
            'table': 'silver_repair_issue',
            'column': 'property_ref',
            'check': 'sql',
            'sql': "SELECT i.issue_id AS record_key, 'property_ref' AS column_name,\n"
                   "       concat('repair on ', p.tenure, ' home ', i.property_ref) AS "
                   'detail\n'
                   'FROM {db}.silver_repair_issue i\n'
                   'JOIN {db}.silver_property p ON p.property_ref = i.property_ref\n'
                   "WHERE p.tenure <> 'PRS'\n"},
           {'id': 'DQ074',
            'name': 'Certificates are only on rental (PRS) homes',
            'dimension': 'consistency',
            'severity': 'medium',
            'owner': 'Compliance Manager',
            'table': 'silver_compliance_certificate',
            'column': 'property_ref',
            'check': 'sql',
            'sql': "SELECT c.certificate_id AS record_key, 'property_ref' AS column_name,\n"
                   "       concat(c.certificate_type, ' on ', p.tenure, ' home ', "
                   'c.property_ref) AS detail\n'
                   'FROM {db}.silver_compliance_certificate c\n'
                   'JOIN {db}.silver_property p ON p.property_ref = c.property_ref\n'
                   "WHERE p.tenure <> 'PRS'\n"},
           {'id': 'DQ075',
            'name': 'Gas Safety certificates only where the home has gas',
            'dimension': 'consistency',
            'severity': 'high',
            'owner': 'Compliance Manager',
            'table': 'silver_compliance_certificate',
            'column': 'certificate_type',
            'check': 'sql',
            'sql': "SELECT c.certificate_id AS record_key, 'certificate_type' AS "
                   'column_name,\n'
                   "       concat('Gas Safety Certificate on ', c.property_ref, ', which has "
                   "no gas supply') AS detail\n"
                   'FROM {db}.silver_compliance_certificate c\n'
                   'JOIN {db}.silver_property p ON p.property_ref = c.property_ref\n'
                   "WHERE c.certificate_type = 'Gas Safety Certificate' AND NOT p.has_gas\n"},
           {'id': 'DQ076',
            'name': 'Offers are accepted only on agreed listings',
            'dimension': 'consistency',
            'severity': 'medium',
            'owner': 'Head of Lettings and Sales',
            'table': 'silver_offer',
            'column': 'offer_status',
            'check': 'sql',
            'sql': "SELECT o.offer_id AS record_key, 'offer_status' AS column_name,\n"
                   "       concat('accepted on listing ', o.listing_id, ' which is ', "
                   'l.listing_status,\n'
                   "              CASE WHEN l.agreed_date IS NULL THEN ' with no agreed date' "
                   "ELSE '' END) AS detail\n"
                   'FROM {db}.silver_offer o\n'
                   'JOIN {db}.silver_listing l ON l.listing_id = o.listing_id\n'
                   "WHERE o.offer_status = 'Accepted'\n"
                   "  AND (l.agreed_date IS NULL OR l.listing_status NOT IN ('Let', "
                   "'Sold'))\n",
            'checked_sql': 'SELECT COUNT(*) FROM {db}.silver_offer WHERE offer_status = '
                           "'Accepted'"},
           {'id': 'DQ077',
            'name': "Deposit within the Tenant Fees Act cap (5 weeks' rent)",
            'dimension': 'consistency',
            'severity': 'high',
            'owner': 'Head of Housing Management',
            'table': 'silver_tenancy',
            'column': 'deposit_amount',
            'check': 'sql',
            'sql': "SELECT tenancy_ref AS record_key, 'deposit_amount' AS column_name,\n"
                   "       concat('deposit ', deposit_amount, ' is over 5 weeks of rent ',\n"
                   '              round(rent_pcm * 12 / 52 * 5, 2)) AS detail\n'
                   'FROM {db}.silver_tenancy\n'
                   "WHERE tenancy_type = 'Assured Shorthold Tenancy'\n"
                   '  AND deposit_amount > round(rent_pcm * 12 / 52 * 5, 2) + 0.01\n',
            'checked_sql': 'SELECT COUNT(*) FROM {db}.silver_tenancy\n'
                           "WHERE tenancy_type = 'Assured Shorthold Tenancy'\n"},
           {'id': 'DQ090',
            'name': 'Silver tables were loaded in the last 24 hours',
            'dimension': 'timeliness',
            'severity': 'medium',
            'owner': 'table owner',
            'table': 'all',
            'column': 'loaded_at',
            'check': 'freshness',
            'max_age_hours': 24},
           {'id': 'DQ091',
            'name': 'FixFlo issues have changed in the last 7 days (feed is flowing)',
            'dimension': 'timeliness',
            'severity': 'medium',
            'owner': 'Head of Repairs and Maintenance',
            'table': 'silver_repair_issue',
            'column': 'status_changed_at',
            'check': 'freshness',
            'max_age_hours': 168},
           {'id': 'DQ092',
            'name': 'Rent ledger has transactions in the last 35 days (feed is flowing)',
            'dimension': 'timeliness',
            'severity': 'medium',
            'owner': 'Head of Finance',
            'table': 'silver_rent_transaction',
            'column': 'transaction_date',
            'check': 'freshness',
            'max_age_hours': 840}]}
# --- end generated ---

## Turning rules into SQL
Every check becomes two queries on one Silver table: one that returns the failing rows
(`record_key`, `column_name`, `detail`) and one that counts the records checked.

In [ ]:
from datetime import UTC, datetime

checked_at = datetime.now(UTC).replace(tzinfo=None)
run_id = run_id or f"DQ{checked_at:%Y%m%dT%H%M%S}"
db = silver_db

TABLE_OWNER = {
    table: domain["owner"] for domain in DQ_RULES["domains"].values() for table in domain["tables"]
}


def q(name: str) -> str:
    return f"`{name}`"


def literal(text: str) -> str:
    """A Spark SQL string literal (backslashes and quotes escaped)."""
    return "'" + str(text).replace("\\", "\\\\").replace("'", "\\'") + "'"


def record_key(table: str, alias: str = "") -> str:
    prefix = f"{alias}." if alias else ""
    cols = ", ".join(f"CAST({prefix}{q(k)} AS STRING)" for k in SILVER_CONTRACT[table]["key"])
    return f"concat_ws('|', {cols})"


def checks_for(rule: dict) -> list[dict]:
    """One entry per table/column the rule looks at: {table, column, failing, checked}."""
    check, t = rule["check"], rule["table"]
    tables = list(SILVER_CONTRACT) if t == "all" else [t]
    out = []

    if check == "primary_key":
        for table in tables:
            key = SILVER_CONTRACT[table]["key"]
            blank = " OR ".join(f"{q(k)} IS NULL" for k in key)
            out.append(
                {
                    "table": table,
                    "column": ", ".join(key),
                    "failing": f"""
                        SELECT {record_key(table)} AS record_key, {literal(", ".join(key))} AS column_name,
                               CASE WHEN {blank} THEN 'key is blank'
                                    ELSE concat('key appears ', n, ' times') END AS detail
                        FROM (SELECT *, COUNT(*) OVER (PARTITION BY {", ".join(q(k) for k in key)}) AS n
                              FROM {db}.{table}) x
                        WHERE n > 1 OR {blank}""",
                    "checked": f"SELECT COUNT(*) FROM {db}.{table}",
                }
            )

    elif check == "foreign_key":
        for table in tables:
            for column, ref in SILVER_CONTRACT[table]["foreign_keys"]:
                ref_key = SILVER_CONTRACT[ref]["key"][0]
                out.append(
                    {
                        "table": table,
                        "column": column,
                        "failing": f"""
                            SELECT {record_key(table, "c")} AS record_key, '{column}' AS column_name,
                                   concat('{column} ', c.{q(column)}, ' not found in {ref}') AS detail
                            FROM {db}.{table} c
                            LEFT ANTI JOIN {db}.{ref} r ON r.{q(ref_key)} = c.{q(column)}
                            WHERE c.{q(column)} IS NOT NULL""",
                        "checked": f"SELECT COUNT(*) FROM {db}.{table} WHERE {q(column)} IS NOT NULL",
                    }
                )

    elif check == "type_converts":
        # Silver logs every value it couldn't convert; only the table's latest load counts.
        for table in tables:
            out.append(
                {
                    "table": table,
                    "column": "typed columns",
                    "failing": f"""
                        SELECT record_key, column_name,
                               concat("'", raw_value, "' is not a valid ", target_type) AS detail
                        FROM {db}._cast_errors
                        WHERE table_name = '{table}'
                          AND loaded_at = (SELECT max(loaded_at) FROM {db}.{table})""",
                    "checked": f"SELECT COUNT(*) FROM {db}.{table}",
                }
            )

    elif check == "not_null":
        cols = rule["columns"]
        out.append(
            {
                "table": t,
                "column": ", ".join(cols),
                "failing": " UNION ALL ".join(
                    f"""SELECT {record_key(t)} AS record_key, '{c}' AS column_name,
                               '{c} is blank' AS detail
                        FROM {db}.{t}
                        WHERE {q(c)} IS NULL OR trim(CAST({q(c)} AS STRING)) = ''""" for c in cols
                ),
                "checked": f"SELECT COUNT(*) * {len(cols)} FROM {db}.{t}",
            }
        )

    elif check in ("pattern", "allowed_values", "range"):
        c = rule["column"]
        if check == "pattern":
            bad = f"NOT ({q(c)} RLIKE {literal(rule['regex'])})"
            why = f'concat("\'", {q(c)}, "\' is not in the expected format")'
        elif check == "allowed_values":
            bad = f"{q(c)} NOT IN ({', '.join(literal(v) for v in rule['values'])})"
            why = f'concat("\'", {q(c)}, "\' is not an allowed value")'
        else:
            bad = f"({q(c)} < {rule['min']} OR {q(c)} > {rule['max']})"
            why = f"concat({q(c)}, ' is outside {rule['min']} to {rule['max']}')"
        out.append(
            {
                "table": t,
                "column": c,
                "failing": f"""
                    SELECT {record_key(t)} AS record_key, '{c}' AS column_name, {why} AS detail
                    FROM {db}.{t}
                    WHERE {q(c)} IS NOT NULL AND {bad}""",
                "checked": f"SELECT COUNT(*) FROM {db}.{t} WHERE {q(c)} IS NOT NULL",
            }
        )

    elif check == "date_window":
        for c in rule["columns"]:
            out.append(
                {
                    "table": t,
                    "column": c,
                    "failing": f"""
                        SELECT {record_key(t)} AS record_key, '{c}' AS column_name,
                               concat('{c} ', {q(c)}, ' is before {rule['min']} or in the future')
                                   AS detail
                        FROM {db}.{t}
                        WHERE CAST({q(c)} AS DATE) < DATE '{rule['min']}'
                           OR CAST({q(c)} AS DATE) > current_date()""",
                    "checked": f"SELECT COUNT(*) FROM {db}.{t} WHERE {q(c)} IS NOT NULL",
                }
            )

    elif check == "freshness":
        c, hours = rule["column"], rule["max_age_hours"]
        for table in tables:
            out.append(
                {
                    "table": table,
                    "column": c,
                    "failing": f"""
                        SELECT '{table}' AS record_key, '{c}' AS column_name,
                               concat('newest {c} is ', coalesce(CAST(newest AS STRING), 'missing'),
                                      ', more than {hours} hours ago') AS detail
                        FROM (SELECT max(CAST({q(c)} AS TIMESTAMP)) AS newest FROM {db}.{table}) x
                        WHERE newest IS NULL
                           OR newest < current_timestamp() - INTERVAL {hours} HOURS""",
                    "checked": "SELECT 1",
                }
            )

    elif check == "sql":
        out.append(
            {
                "table": t,
                "column": rule["column"],
                "failing": rule["sql"].format(db=db),
                "checked": rule.get("checked_sql", f"SELECT COUNT(*) FROM {{db}}.{t}").format(
                    db=db
                ),
            }
        )

    else:
        raise ValueError(f"{rule['id']}: unknown check {check!r}")
    return out

## Run every rule

In [ ]:
from pyspark.sql import types as T

for ddl in (
    f"""CREATE TABLE IF NOT EXISTS {db}.dq_results (
            run_id STRING, rule_id STRING, rule_name STRING, dimension STRING,
            severity STRING, owner STRING, table_name STRING, column_name STRING,
            records_checked BIGINT, failures BIGINT, failure_pct DOUBLE, result STRING,
            sample_failing_keys STRING, error STRING, checked_at TIMESTAMP) USING DELTA""",
    f"""CREATE TABLE IF NOT EXISTS {db}.dq_exceptions (
            run_id STRING, rule_id STRING, rule_name STRING, dimension STRING,
            severity STRING, owner STRING, table_name STRING, record_key STRING,
            column_name STRING, detail STRING, checked_at TIMESTAMP) USING DELTA""",
):
    spark.sql(ddl)
# Re-running the same run_id replaces its rows.
spark.sql(f"DELETE FROM {db}.dq_results WHERE run_id = '{run_id}'")
spark.sql(f"DELETE FROM {db}.dq_exceptions WHERE run_id = '{run_id}'")

results = []
for rule in DQ_RULES["rules"]:
    for chk in checks_for(rule):
        owner = TABLE_OWNER.get(chk["table"]) if rule["owner"] == "table owner" else rule["owner"]
        row = {
            "run_id": run_id,
            "rule_id": rule["id"],
            "rule_name": rule["name"],
            "dimension": rule["dimension"],
            "severity": rule["severity"],
            "owner": owner,
            "table_name": chk["table"],
            "column_name": chk["column"],
            "checked_at": checked_at,
        }
        try:
            failing = spark.sql(chk["failing"])
            failing.createOrReplaceTempView("failing_rows")
            spark.sql(f"""
                INSERT INTO {db}.dq_exceptions
                SELECT {literal(run_id)}, {literal(rule["id"])}, {literal(rule["name"])},
                       {literal(rule["dimension"])}, {literal(rule["severity"])},
                       {literal(owner)}, {literal(chk["table"])},
                       record_key, column_name, detail, TIMESTAMP '{checked_at}'
                FROM failing_rows
            """)
            failures = spark.table("failing_rows").count()
            checked = spark.sql(chk["checked"]).first()[0]
            sample = [r[0] for r in failing.select("record_key").limit(10).collect()]
            row.update(
                records_checked=checked,
                failures=failures,
                failure_pct=round(100 * failures / checked, 3) if checked else 0.0,
                result="PASS" if failures == 0 else "FAIL",
                sample_failing_keys=", ".join(str(k) for k in sample) or None,
                error=None,
            )
        except Exception as exc:  # noqa: BLE001 - a broken rule must not stop the others
            row.update(
                records_checked=None,
                failures=None,
                failure_pct=None,
                result="ERROR",
                sample_failing_keys=None,
                error=str(exc)[:1000],
            )
        results.append(row)

RESULT_SCHEMA = spark.table(f"{db}.dq_results").schema
spark.createDataFrame(
    [tuple(r[f.name] for f in RESULT_SCHEMA.fields) for r in results], RESULT_SCHEMA
).write.mode("append").saveAsTable(f"{db}.dq_results")

## Null profile
How many values are blank in every Silver column. Not a pass/fail rule: it shows where
the data is thin, so the owners can decide which fields must be filled.

In [ ]:
profile_rows = []
for table, spec in SILVER_CONTRACT.items():
    cols = [c for c, _ in spec["columns"]]
    counts = spark.sql(
        "SELECT COUNT(*) AS total_rows, "
        + ", ".join(
            f"SUM(CASE WHEN {q(c)} IS NULL OR trim(CAST({q(c)} AS STRING)) = '' "
            f"THEN 1 ELSE 0 END) AS {q(c)}"
            for c in cols
        )
        + f" FROM {db}.{table}"
    ).first()
    total = counts["total_rows"]
    for c in cols:
        blank = counts[c] or 0
        profile_rows.append(
            (run_id, table, c, total, blank, round(100 * blank / total, 2) if total else 0.0)
        )

PROFILE_SCHEMA = T.StructType(
    [
        T.StructField("run_id", T.StringType()),
        T.StructField("table_name", T.StringType()),
        T.StructField("column_name", T.StringType()),
        T.StructField("total_rows", T.LongType()),
        T.StructField("blank_values", T.LongType()),
        T.StructField("blank_pct", T.DoubleType()),
    ]
)
spark.sql(
    f"CREATE TABLE IF NOT EXISTS {db}.dq_null_profile ("
    "run_id STRING, table_name STRING, column_name STRING, total_rows BIGINT, "
    "blank_values BIGINT, blank_pct DOUBLE) USING DELTA"
)
spark.sql(f"DELETE FROM {db}.dq_null_profile WHERE run_id = '{run_id}'")
spark.createDataFrame(profile_rows, PROFILE_SCHEMA).write.mode("append").saveAsTable(
    f"{db}.dq_null_profile"
)

## Summary

In [ ]:
failed = [r for r in results if r["result"] == "FAIL"]
errors = [r for r in results if r["result"] == "ERROR"]
print(
    f"DQ run {run_id}: {len(results)} checks from {len(DQ_RULES['rules'])} rules; "
    f"{len(results) - len(failed) - len(errors)} passed, {len(failed)} failed, "
    f"{len(errors)} could not run"
)
for r in sorted(failed, key=lambda r: ("high", "medium", "low").index(r["severity"])):
    print(
        f"  FAIL {r['rule_id']} [{r['severity']}] {r['table_name']}.{r['column_name']}: "
        f"{r['failures']} of {r['records_checked']} – {r['rule_name']}"
    )
for r in errors:
    print(f"  ERROR {r['rule_id']} {r['table_name']}: {r['error'][:300]}")
if errors:
    raise RuntimeError(f"{len(errors)} checks could not run; see dq_results.error")